# Extract Embeddings from Pre-trained Nicheformer Model

This notebook extracts embeddings from a pre-trained Nicheformer model and stores them in an AnnData object.

In [ ]:
import os
import numpy as np
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
import torch
from torch.utils.data import DataLoader
import anndata as ad
from tqdm import tqdm

from nicheformer.models import Nicheformer
from nicheformer.data import NicheformerDataset



## Configuration

Set up the configuration parameters for the embedding extraction.

In [ ]:
config = {
    'data_path': '/media/rokny/DATA2/Sally/data/srt/mouse_hyppocampus_slideseqv2/sshippo.h5ad', # Path to your AnnData file
    'technology_mean_path': '/media/rokny/DATA2/Sally/Nicheformer/nicheformer/data/model_means/MH_mean_script.npy', # Path to technology mean file
    'checkpoint_path': '/media/rokny/DATA2/Sally/Nicheformer/nicheformer/nicheformer.ckpt',  # Path to model checkpoint
    'output_path': '/media/rokny/DATA2/Sally/Nicheformer/nicheformer/data/benchmarking/MH_continual_pretrain_with_embeddings.h5ad',  # Where to save the result, it is a new h5ad
    'output_dir': '.',  # Directory for any intermediate outputs
    'batch_size': 9,
    'max_epochs': 2,
    'max_seq_len': 1500, 
    'aux_tokens': 30, 
    'chunk_size': 1000, # to prevent OOM
    'num_workers': 4,
    'precision': 'bf16-mixed',
    'embedding_layer': -1,  # Which layer to extract embeddings from (-1 for last layer)
    'embedding_name': 'embeddings'  # Name suffix for the embedding key in adata.obsm
}

## Load Data and Create Dataset

In [ ]:
model = ad.read_h5ad('/media/rokny/DATA2/Sally/Nicheformer/nicheformer/data/model_means/model.h5ad') 


In [ ]:
# Set random seed for reproducibility
pl.seed_everything(42)

# Load data
adata = ad.read_h5ad(config['data_path'])
technology_mean = np.load(config['technology_mean_path'])

In [ ]:
# Convert gene symbols to ensembl IDs
import mygene
import pandas as pd

mg = mygene.MyGeneInfo()

gene_symbols = adata.var_names.tolist()

out = mg.querymany(
    gene_symbols,
    scopes="symbol", 
    fields="ensembl.gene",
    species="human" 
)

df = pd.DataFrame(out)

df["ensembl_id"] = df["ensembl"].apply(
    lambda x: x[0]["gene"] if isinstance(x, list) else (x["gene"] if isinstance(x, dict) else None)
)

# Build mapping dictionary: {symbol -> ensembl_id}
mapping = df.set_index("query")["ensembl_id"].dropna().to_dict()

# Replace var_names
adata.var["gene_symbol"] = adata.var_names  # keep original symbols
adata.var_names = [mapping.get(g, g) for g in adata.var_names]
adata.var_names_make_unique()

common_genes = model.var_names.intersection(adata.var_names)

adata = adata[:, common_genes].copy()

for key in list(adata.obsm.keys()):
    if key != 'spatial':
        del adata.obsm[key]

# format data properly with the model
adata = ad.concat([model, adata], join='outer', axis=0)
# dropping the first observation 
adata = adata[1:].copy()

In [ ]:
adata

As a reference, the metadata tokens are 

modality_dict = {
    'dissociated': 3,
    'spatial': 4,}

specie_dict = {
    'human': 5,
    'Homo sapiens': 5,
    'Mus musculus': 6,
    'mouse': 6,}

technology_dict = {
    "merfish": 7,
    "MERFISH": 7,
    "cosmx": 8,
    "NanoString digital spatial profiling": 8,
    "visium": 9,
    "10x 5' v2": 10,
    "10x 3' v3": 11,
    "10x 3' v2": 12,
    "10x 5' v1": 13,
    "10x 3' v1": 14,
    "10x 3' transcription profiling": 15, 
    "10x transcription profiling": 15,
    "10x 5' transcription profiling": 16,
    "CITE-seq": 17, 
    "Smart-seq v4": 18,
}

In [ ]:
# Change accordingly

adata.obs['modality'] = 4 
adata.obs['specie'] = 6 
adata.obs['assay'] = 19 # slide-seq v2

In [ ]:
adata.obs['nicheformer_split'] = 'train'

train_samples_count = (adata.obs['nicheformer_split'] == 'train').sum()
print(f"Number of samples labeled 'train': {train_samples_count}")

In [ ]:
# Create dataset
dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=1500,
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000),
    metadata_fields={'obs': ['modality', 'specie', 'assay']}
)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

## Load Model and Set Up Trainer

In [ ]:
# Load pre-trained model
model = Nicheformer.load_from_checkpoint(checkpoint_path=config['checkpoint_path'], strict=False)
# model.eval()  # Set to evaluation mode

checkpoint_callback = ModelCheckpoint(dirpath=config['checkpoint_path'], every_n_train_steps=1500, monitor='train_loss', save_top_k=-1)
lr_monitor = LearningRateMonitor(logging_interval='step')


# Configure trainer
trainer = pl.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    max_epochs=config['max_epochs'],
    log_every_n_steps=100,
    check_val_every_n_epoch=50,
    default_root_dir=config['output_dir'],
    precision=config['precision'],
    callbacks=[checkpoint_callback, lr_monitor],
    gradient_clip_val=1,
    accumulate_grad_batches=10
)

In [ ]:
print(f"Training model from checkpoint!")
trainer.fit(model=model, train_dataloaders=dataloader)


## Extract Embeddings

In [ ]:
print("Extracting embeddings...")
embeddings = []
device = model.embeddings.weight.device

with torch.no_grad():
    for batch in tqdm(dataloader):
        # Move batch to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()}

        # Get embeddings from the model
        emb = model.get_embeddings(
            batch=batch,
            layer=config.get('embedding_layer', -1)  # Default to last layer
        )
        embeddings.append(emb.cpu().numpy())


# Concatenate all embeddings
embeddings = np.concatenate(embeddings, axis=0)

## Save Results

In [ ]:
# Store embeddings in AnnData object
embedding_key = f"X_niche_{config.get('embedding_name', 'embeddings')}"
adata.obsm[embedding_key] = embeddings
print(embedding_key)

# # Save updated AnnData
adata.write_h5ad(config['output_path'])

print(f"Embeddings saved to {config['output_path']} in obsm['{embedding_key}']")

## Spatial Plot

Run the import block at top of notebook first.

In [ ]:
# Set a seed for reproducibility

import random

def fix_seed(seed):
    # Fix for Python hash seed (to ensure reproducibility in Python operations)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix for random seed (used by random module)
    random.seed(seed)
    
    # Fix for numpy random operations
    np.random.seed(seed)
    
    # Fix for PyTorch random seed (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional configuration for CUBLAS for reproducibility
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # For scanpy or any random process in other libraries
    # We can use a random_state argument where applicable, like in scanpy PCA, DEG, etc.

# Set the seed
seed = 42
fix_seed(seed)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Read the AnnData objects
adata = sc.read_h5ad("/media/rokny/DATA2/Sally/Nicheformer/nicheformer/data/benchmarking/MH_continual_pretrain_with_embeddings.h5ad")

## Clustering

In [ ]:
import numpy as np
import pandas as pd
from sklearn import metrics
import scanpy as sc
from sklearn.decomposition import PCA


def clustering(adata, n_clusters=7, key='emb', method='leiden', start=0.1, end=3.0, increment=0.01):
    """\
    Spatial clustering based the learned representation.

    Returns
    -------
    None.

    """
    
    pca = PCA(n_components=20, random_state=seed) 
    embedding = pca.fit_transform(adata.obsm[key].copy())
    adata.obsm['emb_pca'] = embedding
    
    if method == 'leiden':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.leiden(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['leiden']
    elif method == 'louvain':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.louvain(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['louvain'] 
       
    

def extract_top_value(map_matrix, retain_percent = 0.1): 
    '''\
    Filter out cells with low mapping probability

    Parameters
    ----------
    map_matrix : array
        Mapped matrix with m spots and n cells.
    retain_percent : float, optional
        The percentage of cells to retain. The default is 0.1.

    Returns
    -------
    output : array
        Filtered mapped matrix.

    '''

    #retain top 1% values for each spot
    top_k  = retain_percent * map_matrix.shape[1]
    output = map_matrix * (np.argsort(np.argsort(map_matrix)) >= map_matrix.shape[1] - top_k)
    
    return output 
    
def search_res(adata, n_clusters, method='leiden', use_rep='emb', start=0.1, end=3.0, increment=0.01):
    '''\
    Searching corresponding resolution according to given cluster number
    
    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Targetting number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.    
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float 
        The end value for searching.
    increment : float
        The step size to increase.
        
    Returns
    -------
    res : float
        Resolution.
        
    '''
    print('Searching resolution...')
    label = 0
    sc.pp.neighbors(adata, n_neighbors=50, use_rep=use_rep, random_state=seed)
    for res in sorted(list(np.arange(start, end, increment)), reverse=True):
        if method == 'leiden':
           sc.tl.leiden(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['leiden']).leiden.unique())
           print('resolution={}, cluster number={}'.format(res, count_unique))
        elif method == 'louvain':
           sc.tl.louvain(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['louvain']).louvain.unique()) 
           print('resolution={}, cluster number={}'.format(res, count_unique))
        if count_unique == n_clusters:
            label = 1
            break

    assert label==1, "Resolution is not found. Please try bigger range or smaller step!." 
       
    return res    


In [ ]:
# Run Leiden clustering 

n_clusters = 14
tool='leiden'

clustering(adata, n_clusters, key='X_niche_embeddings', method=tool, start=0.1, end=0.7, increment=0.01)

In [ ]:
labels = adata.obs['leiden'].astype(int)

In [ ]:
p=['#1f77b4',
 '#aec7e8',
 '#ff7f0e',
 '#ffbb78',
 '#2ca02c',
 '#98df8a',
 '#d62728',
 '#ff9896',
 '#9467bd',
 '#c5b0d5',
 '#8c564b',
 '#c49c94',
 '#e377c2',
 '#f7b6d2']

In [ ]:
import matplotlib.pyplot as plt

Model_name='nicheformer'
step='continualpretrain' # or 'zero_shot'
dataset='MH' # or 'BC' or 'mouse_slideseq'

plt.rcParams["figure.figsize"] = (3,3)
sc.pl.embedding(adata, basis="spatial", color="domain", palette=p, show=False, title='')
plt.gca().invert_yaxis()
plt.axis('off')
plt.savefig(f"/media/rokny/DATA2/Sally/Nicheformer/figures/{dataset}_{Model_name}_leiden_{step}.svg",bbox_inches="tight")
plt.show()

## ARI, NMI & Silhouette Scores

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

ari_score = adjusted_rand_score(adata.obs['leiden'].to_numpy(), adata.obs['cluster'].to_numpy())
nmi_score = normalized_mutual_info_score(adata.obs['leiden'].to_numpy(), adata.obs['cluster'].to_numpy())
sil_score = silhouette_score(adata.obsm['X_niche_embeddings'], adata.obs['leiden'].astype(int))

print(f"'ari': {ari_score}, 'nmi': {nmi_score}, 'sil': {sil_score}")

## Save results to npz file

In [ ]:
import numpy as np

ARI, NMI, SIL = float(ari_score), float(nmi_score), float(sil_score)

np.savez_compressed(
    f"/media/rokny/DATA2/Sally/Nicheformer/benchmarking_results/{Model_name}_{step}_clusters_{dataset}.npz",
    labels=labels,      
    embeddings=adata.obsm['X_niche_embeddings'],
    ARI=ARI, NMI=NMI, SIL=SIL,
)